# Projet LinkedIn - Analyse des Offres d'Emploi avec Snowflake

Ce notebook contient l'ensemble du code SQL et Python du projet.
Il suit une architecture Medallion en 3 couches :
- Bronze : données brutes depuis S3
- Silver : données nettoyées et typées
- Gold   : données agregées prêtes pour l'analyse

---
## ETAPE 1 - Creation de la base de données et des schemas

In [ ]:
-- Creation de la base de données principale
CREATE DATABASE linkedin;

USE DATABASE linkedin;

-- Creation des 3 schemas selon l'architecture Medallion
-- Bronze : données brutes telles quelles depuis S3
CREATE SCHEMA linkedin.bronze;

-- Silver : données nettoyées, typées et dédupliquées
CREATE SCHEMA linkedin.silver;

-- Gold : données agrégées prêtes pour les analyses et visualisations
CREATE SCHEMA linkedin.gold;

-- Vérification des schemas créés
SHOW SCHEMAS IN DATABASE linkedin;

---
## ETAPE 2 - Creation du stage S3 et des formats de fichiers

In [ ]:
USE SCHEMA linkedin.bronze;

-- Creation du stage externe pointant vers le bucket S3 public
-- Le stage est une zone de transit entre S3 et Snowflake
CREATE OR REPLACE STAGE linkedin.bronze.s3_stage
  URL = 's3://snowflake-lab-bucket/';

-- Format pour les fichiers CSV
-- FIELD_OPTIONALLY_ENCLOSED_BY : gère les champs contenant des virgules
-- SKIP_HEADER : ignore la première ligne (noms des colonnes)
-- NULL_IF : convertit les valeurs textuelles nulles en NULL Snowflake
CREATE OR REPLACE FILE FORMAT linkedin.bronze.csv_format
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER = 1
  NULL_IF = ('NULL', 'null', '')
  EMPTY_FIELD_AS_NULL = TRUE
  DATE_FORMAT = 'AUTO'
  TIMESTAMP_FORMAT = 'AUTO';

-- Format pour les fichiers JSON
-- STRIP_OUTER_ARRAY : traite chaque élément du tableau JSON séparément
CREATE OR REPLACE FILE FORMAT linkedin.bronze.json_format
  TYPE = 'JSON'
  STRIP_OUTER_ARRAY = TRUE
  NULL_IF = ('NULL', 'null', '');

-- Vérification : liste les fichiers disponibles dans le bucket S3
LIST @linkedin.bronze.s3_stage;

---
## ETAPE 3 - Creation des tables Bronze

En Bronze, tout est stocké en VARCHAR pour les CSV et en VARIANT pour les JSON.
Cela evite les erreurs de type lors du chargement et permet d'analyser
les données brutes avant de choisir les bons types dans Silver.

In [ ]:
USE SCHEMA linkedin.bronze;

-- Table des offres d'emploi
-- Toutes les colonnes en VARCHAR car les timestamps sont au format Unix
-- et certains champs numériques contiennent des valeurs inattendues
CREATE OR REPLACE TABLE linkedin.bronze.job_postings (
    job_id                      VARCHAR,
    company_name                VARCHAR,
    title                       VARCHAR,
    description                 VARCHAR,
    max_salary                  VARCHAR,
    med_salary                  VARCHAR,
    min_salary                  VARCHAR,
    pay_period                  VARCHAR,
    formatted_work_type         VARCHAR,
    location                    VARCHAR,
    applies                     VARCHAR,
    original_listed_time        VARCHAR,
    remote_allowed              VARCHAR,
    views                       VARCHAR,
    job_posting_url             VARCHAR,
    application_url             VARCHAR,
    application_type            VARCHAR,
    expiry                      VARCHAR,
    closed_time                 VARCHAR,
    formatted_experience_level  VARCHAR,
    skills_desc                 VARCHAR,
    listed_time                 VARCHAR,
    posting_domain              VARCHAR,
    sponsored                   VARCHAR,
    work_type                   VARCHAR,
    currency                    VARCHAR,
    compensation_type           VARCHAR
);

-- Table des avantages associés à chaque offre
CREATE OR REPLACE TABLE linkedin.bronze.benefits (
    job_id    VARCHAR,
    inferred  VARCHAR,
    type      VARCHAR
);

-- Table du nombre d'employés et followers par entreprise
CREATE OR REPLACE TABLE linkedin.bronze.employee_counts (
    company_id      VARCHAR,
    employee_count  VARCHAR,
    follower_count  VARCHAR,
    time_recorded   VARCHAR
);

-- Table des compétences associées à chaque offre
CREATE OR REPLACE TABLE linkedin.bronze.job_skills (
    job_id     VARCHAR,
    skill_abr  VARCHAR
);

-- Tables JSON stockées en VARIANT
-- VARIANT peut stocker n'importe quelle structure JSON, même complexe
-- L'extraction des champs se fera dans Silver

-- Informations détaillées sur les entreprises
CREATE OR REPLACE TABLE linkedin.bronze.companies (
    raw_data VARIANT
);

-- Secteurs d'activité par entreprise
CREATE OR REPLACE TABLE linkedin.bronze.company_industries (
    raw_data VARIANT
);

-- Spécialités par entreprise
CREATE OR REPLACE TABLE linkedin.bronze.company_specialities (
    raw_data VARIANT
);

-- Secteurs d'activité par offre d'emploi
CREATE OR REPLACE TABLE linkedin.bronze.job_industries (
    raw_data VARIANT
);

-- Vérification : liste toutes les tables créées dans Bronze
SHOW TABLES IN SCHEMA linkedin.bronze;

---
## ETAPE 4 - Chargement des données dans Bronze

Bonne pratique : tester avec VALIDATION_MODE avant de charger réellement.
- VALIDATION_MODE = 'RETURN_ERRORS' : affiche les erreurs sans charger
- Sans option : charge réellement les données

In [ ]:
USE SCHEMA linkedin.bronze;

-- Chargement des fichiers CSV

-- Chargement des offres d'emploi
COPY INTO linkedin.bronze.job_postings
FROM @linkedin.bronze.s3_stage/job_postings.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.csv_format');

-- Chargement des avantages
COPY INTO linkedin.bronze.benefits
FROM @linkedin.bronze.s3_stage/benefits.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.csv_format');

-- Chargement des compteurs d'employés
COPY INTO linkedin.bronze.employee_counts
FROM @linkedin.bronze.s3_stage/employee_counts.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.csv_format');

-- Chargement des compétences
COPY INTO linkedin.bronze.job_skills
FROM @linkedin.bronze.s3_stage/job_skills.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.csv_format');

-- Chargement des fichiers JSON
-- Les données JSON sont stockées telles quelles dans la colonne VARIANT

-- Chargement des entreprises
COPY INTO linkedin.bronze.companies
FROM @linkedin.bronze.s3_stage/companies.json
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.json_format');

-- Chargement des secteurs par entreprise
COPY INTO linkedin.bronze.company_industries
FROM @linkedin.bronze.s3_stage/company_industries.json
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.json_format');

-- Chargement des spécialités par entreprise
COPY INTO linkedin.bronze.company_specialities
FROM @linkedin.bronze.s3_stage/company_specialities.json
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.json_format');

-- Chargement des secteurs par offre d'emploi
COPY INTO linkedin.bronze.job_industries
FROM @linkedin.bronze.s3_stage/job_industries.json
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.json_format');

-- Vérification : compter les lignes dans chaque table
SELECT 'job_postings' AS table_name, COUNT(*) AS nb_lignes FROM linkedin.bronze.job_postings
UNION ALL SELECT 'benefits', COUNT(*) FROM linkedin.bronze.benefits
UNION ALL SELECT 'companies', COUNT(*) FROM linkedin.bronze.companies
UNION ALL SELECT 'employee_counts', COUNT(*) FROM linkedin.bronze.employee_counts
UNION ALL SELECT 'job_skills', COUNT(*) FROM linkedin.bronze.job_skills
UNION ALL SELECT 'job_industries', COUNT(*) FROM linkedin.bronze.job_industries
UNION ALL SELECT 'company_specialities', COUNT(*) FROM linkedin.bronze.company_specialities
UNION ALL SELECT 'company_industries', COUNT(*) FROM linkedin.bronze.company_industries;

---
## ETAPE 5 - Analyse des données Bronze

Avant de créer les tables Silver, on analyse les données brutes
pour choisir les bons types et identifier les transformations nécessaires.

In [ ]:
-- Analyse de la table benefits
SELECT * FROM linkedin.bronze.benefits LIMIT 5;
SELECT DISTINCT inferred FROM linkedin.bronze.benefits;
SELECT DISTINCT type FROM linkedin.bronze.benefits;
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN job_id IS NULL THEN 1 ELSE 0 END) AS null_job_id,
    SUM(CASE WHEN inferred IS NULL THEN 1 ELSE 0 END) AS null_inferred,
    SUM(CASE WHEN type IS NULL THEN 1 ELSE 0 END) AS null_type
FROM linkedin.bronze.benefits;

In [ ]:
-- Analyse de la table employee_counts
SELECT * FROM linkedin.bronze.employee_counts LIMIT 5;
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN company_id IS NULL THEN 1 ELSE 0 END) AS null_company_id,
    SUM(CASE WHEN employee_count IS NULL THEN 1 ELSE 0 END) AS null_employee_count,
    SUM(CASE WHEN follower_count IS NULL THEN 1 ELSE 0 END) AS null_follower_count,
    SUM(CASE WHEN time_recorded IS NULL THEN 1 ELSE 0 END) AS null_time_recorded
FROM linkedin.bronze.employee_counts;

In [ ]:
-- Analyse de la table job_skills
SELECT * FROM linkedin.bronze.job_skills LIMIT 5;
SELECT DISTINCT skill_abr FROM linkedin.bronze.job_skills LIMIT 20;
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN job_id IS NULL THEN 1 ELSE 0 END) AS null_job_id,
    SUM(CASE WHEN skill_abr IS NULL THEN 1 ELSE 0 END) AS null_skill_abr
FROM linkedin.bronze.job_skills;

In [ ]:
-- Analyse de la table job_postings
-- On vérifie les valeurs distinctes des colonnes catégorielles
-- et le nombre de valeurs nulles par colonne
SELECT * FROM linkedin.bronze.job_postings LIMIT 5;
SELECT DISTINCT pay_period FROM linkedin.bronze.job_postings;
SELECT DISTINCT formatted_work_type FROM linkedin.bronze.job_postings;
SELECT DISTINCT remote_allowed FROM linkedin.bronze.job_postings;
SELECT DISTINCT sponsored FROM linkedin.bronze.job_postings;
SELECT DISTINCT currency FROM linkedin.bronze.job_postings;
SELECT DISTINCT compensation_type FROM linkedin.bronze.job_postings;
SELECT DISTINCT formatted_experience_level FROM linkedin.bronze.job_postings;
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN max_salary IS NULL THEN 1 ELSE 0 END) AS null_max_salary,
    SUM(CASE WHEN min_salary IS NULL THEN 1 ELSE 0 END) AS null_min_salary,
    SUM(CASE WHEN med_salary IS NULL THEN 1 ELSE 0 END) AS null_med_salary,
    SUM(CASE WHEN remote_allowed IS NULL THEN 1 ELSE 0 END) AS null_remote,
    SUM(CASE WHEN applies IS NULL THEN 1 ELSE 0 END) AS null_applies,
    SUM(CASE WHEN views IS NULL THEN 1 ELSE 0 END) AS null_views,
    SUM(CASE WHEN closed_time IS NULL THEN 1 ELSE 0 END) AS null_closed_time,
    SUM(CASE WHEN original_listed_time IS NULL THEN 1 ELSE 0 END) AS null_original_listed_time
FROM linkedin.bronze.job_postings;

In [ ]:
-- Analyse des tables JSON
-- On inspecte la structure brute pour identifier les champs à extraire dans Silver
SELECT raw_data FROM linkedin.bronze.companies LIMIT 3;
SELECT raw_data FROM linkedin.bronze.company_industries LIMIT 3;
SELECT raw_data FROM linkedin.bronze.company_specialities LIMIT 3;
SELECT raw_data FROM linkedin.bronze.job_industries LIMIT 3;

---
## ETAPE 6 - Creation des tables Silver

Les tables Silver ont les bons types déterminés après analyse des données Bronze :
- Les timestamps Unix (millisecondes) → TIMESTAMP
- Les booléens stockés en 0/1 → BOOLEAN
- Les nombres → FLOAT ou INT selon le cas
- Les champs texte → VARCHAR

In [ ]:
USE SCHEMA linkedin.silver;

-- Table job_postings avec les bons types
-- Les timestamps sont en TIMESTAMP (conversion depuis Unix millisecondes dans l'étape 7)
-- remote_allowed et sponsored sont en BOOLEAN (valeurs 0/1 dans le Bronze)
CREATE OR REPLACE TABLE linkedin.silver.job_postings (
    job_id                      VARCHAR,
    company_name                VARCHAR,
    title                       VARCHAR,
    description                 VARCHAR,
    max_salary                  FLOAT,
    med_salary                  FLOAT,
    min_salary                  FLOAT,
    pay_period                  VARCHAR,
    formatted_work_type         VARCHAR,
    location                    VARCHAR,
    applies                     INT,
    original_listed_time        TIMESTAMP,
    remote_allowed              BOOLEAN,
    views                       INT,
    job_posting_url             VARCHAR,
    application_url             VARCHAR,
    application_type            VARCHAR,
    expiry                      TIMESTAMP,
    closed_time                 TIMESTAMP,
    formatted_experience_level  VARCHAR,
    skills_desc                 VARCHAR,
    listed_time                 TIMESTAMP,
    posting_domain              VARCHAR,
    sponsored                   BOOLEAN,
    work_type                   VARCHAR,
    currency                    VARCHAR,
    compensation_type           VARCHAR
);

-- Table benefits
-- inferred en BOOLEAN (valeurs 0/1 dans le Bronze)
CREATE OR REPLACE TABLE linkedin.silver.benefits (
    job_id    VARCHAR,
    inferred  BOOLEAN,
    type      VARCHAR
);

-- Table companies
-- company_size en INT (valeurs 0 à 7 représentant la taille de l'entreprise)
CREATE OR REPLACE TABLE linkedin.silver.companies (
    company_id    VARCHAR,
    name          VARCHAR,
    description   VARCHAR,
    company_size  INT,
    state         VARCHAR,
    country       VARCHAR,
    city          VARCHAR,
    zip_code      VARCHAR,
    address       VARCHAR,
    url           VARCHAR
);

-- Table employee_counts
-- time_recorded en TIMESTAMP (conversion depuis Unix secondes dans l'étape 7)
CREATE OR REPLACE TABLE linkedin.silver.employee_counts (
    company_id      VARCHAR,
    employee_count  INT,
    follower_count  INT,
    time_recorded   TIMESTAMP
);

-- Table job_skills (pas de transformation nécessaire)
CREATE OR REPLACE TABLE linkedin.silver.job_skills (
    job_id     VARCHAR,
    skill_abr  VARCHAR
);

-- Table job_industries
-- industry_id en INT (identifiant numérique du secteur)
CREATE OR REPLACE TABLE linkedin.silver.job_industries (
    job_id       VARCHAR,
    industry_id  INT
);

-- Table company_specialities
CREATE OR REPLACE TABLE linkedin.silver.company_specialities (
    company_id  VARCHAR,
    speciality  VARCHAR
);

-- Table company_industries
CREATE OR REPLACE TABLE linkedin.silver.company_industries (
    company_id  VARCHAR,
    industry    VARCHAR
);

-- Vérification : liste toutes les tables créées dans Silver
SHOW TABLES IN SCHEMA linkedin.silver;

---
## ETAPE 7 - Peuplement des tables Silver

Transformations appliquées :
- Timestamps Unix millisecondes → TIMESTAMP : division par 1000 puis CAST en BIGINT
- applies et views stockés en FLOAT → INT : TRY_CAST
- remote_allowed et sponsored en 0/1 → BOOLEAN : CASE WHEN
- company_name contient des IDs flottants (ex: 7789.0) → nettoyage avec SPLIT_PART
- Doublons dans job_postings → déduplification avec ROW_NUMBER()

In [ ]:
USE SCHEMA linkedin.silver;

-- Chargement de job_postings avec déduplification et transformations
-- ROW_NUMBER() OVER (PARTITION BY job_id) permet de garder une seule ligne par offre
-- TO_TIMESTAMP_NTZ avec BIGINT : conversion correcte des timestamps Unix millisecondes
INSERT INTO linkedin.silver.job_postings
SELECT
    job_id,
    -- Suppression du .0 du company_name (ex: 7789.0 devient 7789)
    SPLIT_PART(company_name, '.', 1)            AS company_name,
    title,
    description,
    TRY_CAST(max_salary AS FLOAT)               AS max_salary,
    TRY_CAST(med_salary AS FLOAT)               AS med_salary,
    TRY_CAST(min_salary AS FLOAT)               AS min_salary,
    pay_period,
    formatted_work_type,
    location,
    TRY_CAST(applies AS INT)                    AS applies,
    -- Conversion Unix millisecondes en TIMESTAMP : division par 1000 puis CAST BIGINT
    TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(original_listed_time) / 1000 AS BIGINT)) AS original_listed_time,
    -- Conversion 1.0/vide en BOOLEAN
    CASE WHEN remote_allowed = '1.0' THEN TRUE ELSE FALSE END AS remote_allowed,
    TRY_CAST(views AS INT)                      AS views,
    job_posting_url,
    application_url,
    application_type,
    TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(expiry) / 1000 AS BIGINT))       AS expiry,
    TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(closed_time) / 1000 AS BIGINT))  AS closed_time,
    formatted_experience_level,
    skills_desc,
    TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(listed_time) / 1000 AS BIGINT))  AS listed_time,
    posting_domain,
    -- Conversion 0/1 en BOOLEAN
    CASE WHEN sponsored = '1' THEN TRUE ELSE FALSE END AS sponsored,
    work_type,
    currency,
    compensation_type
FROM (
    SELECT *,
        -- Numérotation des doublons : on garde uniquement la première occurrence (rn = 1)
        ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY job_id) AS rn
    FROM linkedin.bronze.job_postings
)
WHERE rn = 1;

In [ ]:
-- Chargement de benefits
-- inferred converti de 0/1 en BOOLEAN
INSERT INTO linkedin.silver.benefits
SELECT
    job_id,
    CASE WHEN inferred = '1' THEN TRUE ELSE FALSE END AS inferred,
    type
FROM linkedin.bronze.benefits;

In [ ]:
-- Chargement de companies depuis le JSON VARIANT
-- Extraction des champs avec la notation raw_data:nom_champ::TYPE
-- NULLIF nettoie les valeurs '0' dans state et zip_code (données incorrectes)
INSERT INTO linkedin.silver.companies
SELECT
    raw_data:company_id::VARCHAR                AS company_id,
    raw_data:name::VARCHAR                      AS name,
    raw_data:description::VARCHAR               AS description,
    raw_data:company_size::INT                  AS company_size,
    NULLIF(raw_data:state::VARCHAR, '0')        AS state,
    raw_data:country::VARCHAR                   AS country,
    raw_data:city::VARCHAR                      AS city,
    NULLIF(raw_data:zip_code::VARCHAR, '0')     AS zip_code,
    raw_data:address::VARCHAR                   AS address,
    raw_data:url::VARCHAR                       AS url
FROM linkedin.bronze.companies;

In [ ]:
-- Chargement de employee_counts
-- time_recorded en secondes Unix (pas millisecondes) donc pas de division par 1000
INSERT INTO linkedin.silver.employee_counts
SELECT
    company_id,
    TRY_CAST(employee_count AS INT)                                        AS employee_count,
    TRY_CAST(follower_count AS INT)                                        AS follower_count,
    TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(time_recorded) AS BIGINT))        AS time_recorded
FROM linkedin.bronze.employee_counts;

In [ ]:
-- Chargement de job_skills (pas de transformation nécessaire)
INSERT INTO linkedin.silver.job_skills
SELECT job_id, skill_abr
FROM linkedin.bronze.job_skills;

-- Chargement de job_industries depuis le JSON VARIANT
-- industry_id converti en INT
INSERT INTO linkedin.silver.job_industries
SELECT
    raw_data:job_id::VARCHAR    AS job_id,
    raw_data:industry_id::INT   AS industry_id
FROM linkedin.bronze.job_industries;

-- Chargement de company_specialities depuis le JSON VARIANT
INSERT INTO linkedin.silver.company_specialities
SELECT
    raw_data:company_id::VARCHAR    AS company_id,
    raw_data:speciality::VARCHAR    AS speciality
FROM linkedin.bronze.company_specialities;

-- Chargement de company_industries depuis le JSON VARIANT
INSERT INTO linkedin.silver.company_industries
SELECT
    raw_data:company_id::VARCHAR    AS company_id,
    raw_data:industry::VARCHAR      AS industry
FROM linkedin.bronze.company_industries;

-- Vérification : compter les lignes dans chaque table Silver
SELECT 'job_postings' AS table_name, COUNT(*) AS nb_lignes FROM linkedin.silver.job_postings
UNION ALL SELECT 'benefits', COUNT(*) FROM linkedin.silver.benefits
UNION ALL SELECT 'companies', COUNT(*) FROM linkedin.silver.companies
UNION ALL SELECT 'employee_counts', COUNT(*) FROM linkedin.silver.employee_counts
UNION ALL SELECT 'job_skills', COUNT(*) FROM linkedin.silver.job_skills
UNION ALL SELECT 'job_industries', COUNT(*) FROM linkedin.silver.job_industries
UNION ALL SELECT 'company_specialities', COUNT(*) FROM linkedin.silver.company_specialities
UNION ALL SELECT 'company_industries', COUNT(*) FROM linkedin.silver.company_industries;

---
## ETAPE 8 - Tests de qualité des données Silver

On vérifie 4 types de problèmes :
1. Doublons
2. Valeurs nulles sur les colonnes critiques
3. Cohérence des données (salaires, tailles, dates)
4. Clés étrangères orphelines

In [ ]:
USE SCHEMA linkedin.silver;

-- Test 1 : Doublons dans job_postings
-- Un job_id ne doit apparaître qu'une seule fois
SELECT job_id, COUNT(*) AS nb
FROM linkedin.silver.job_postings
GROUP BY job_id
HAVING COUNT(*) > 1;

-- Test 1 : Doublons dans companies
-- Un company_id ne doit apparaître qu'une seule fois
SELECT company_id, COUNT(*) AS nb
FROM linkedin.silver.companies
GROUP BY company_id
HAVING COUNT(*) > 1;

In [ ]:
-- Test 2 : Valeurs nulles sur les colonnes critiques
SELECT
    SUM(CASE WHEN job_id IS NULL THEN 1 ELSE 0 END)       AS null_job_id,
    SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END)        AS null_title,
    SUM(CASE WHEN company_name IS NULL THEN 1 ELSE 0 END) AS null_company
FROM linkedin.silver.job_postings;

In [ ]:
-- Test 3 : Cohérence des données

-- Les salaires ne doivent pas être négatifs
SELECT COUNT(*) AS salaires_negatifs
FROM linkedin.silver.job_postings
WHERE max_salary < 0 OR min_salary < 0 OR med_salary < 0;

-- Le salaire minimum doit être inférieur ou égal au salaire maximum
SELECT COUNT(*) AS incoherence_salaire
FROM linkedin.silver.job_postings
WHERE min_salary > max_salary;

-- company_size doit être entre 0 et 7
SELECT COUNT(*) AS taille_invalide
FROM linkedin.silver.companies
WHERE company_size NOT BETWEEN 0 AND 7;

-- Les dates de publication doivent être après 2020
SELECT COUNT(*) AS dates_invalides
FROM linkedin.silver.job_postings
WHERE listed_time < '2020-01-01'
OR listed_time > CURRENT_TIMESTAMP();

In [ ]:
-- Test 4 : Clés étrangères orphelines

-- Les company_id dans company_industries doivent exister dans companies
SELECT COUNT(*) AS orphelins_company_industries
FROM linkedin.silver.company_industries ci
LEFT JOIN linkedin.silver.companies c ON ci.company_id = c.company_id
WHERE c.company_id IS NULL;

-- Les job_id dans benefits doivent exister dans job_postings
SELECT COUNT(*) AS orphelins_benefits
FROM linkedin.silver.benefits b
LEFT JOIN linkedin.silver.job_postings jp ON b.job_id = jp.job_id
WHERE jp.job_id IS NULL;

-- Les job_id dans job_skills doivent exister dans job_postings
SELECT COUNT(*) AS orphelins_job_skills
FROM linkedin.silver.job_skills js
LEFT JOIN linkedin.silver.job_postings jp ON js.job_id = jp.job_id
WHERE jp.job_id IS NULL;

---
## ETAPE 9 - Creation des tables Gold

Les tables Gold contiennent les données pré-agrégées pour chaque analyse.
Avantage : le code Streamlit est plus simple et les requêtes plus rapides
car les jointures complexes sont déjà effectuées.

In [ ]:
USE SCHEMA linkedin.gold;

-- Table 1 : Top 10 des titres de postes les plus publiés par industrie
-- Jointure job_postings → company_industries via company_id
-- ROW_NUMBER() + QUALIFY : garde uniquement le top 10 par industrie
CREATE OR REPLACE TABLE linkedin.gold.top_titres_par_industrie AS
SELECT
    ci.industry,
    jp.title,
    COUNT(*) AS nb_offres,
    ROW_NUMBER() OVER (PARTITION BY ci.industry ORDER BY COUNT(*) DESC) AS rang
FROM linkedin.silver.job_postings jp
JOIN linkedin.silver.company_industries ci
    ON jp.company_name = ci.company_id
GROUP BY ci.industry, jp.title
QUALIFY rang <= 10
ORDER BY ci.industry, nb_offres DESC;

-- Table 2 : Top 10 des postes les mieux rémunérés par industrie
-- AVG(max_salary) : salaire moyen maximum par poste
-- WHERE max_salary IS NOT NULL : exclut les offres sans salaire renseigné
CREATE OR REPLACE TABLE linkedin.gold.top_salaires_par_industrie AS
SELECT
    ci.industry,
    jp.title,
    ROUND(AVG(jp.max_salary), 2) AS salaire_moyen,
    ROW_NUMBER() OVER (PARTITION BY ci.industry ORDER BY AVG(jp.max_salary) DESC) AS rang
FROM linkedin.silver.job_postings jp
JOIN linkedin.silver.company_industries ci
    ON jp.company_name = ci.company_id
WHERE jp.max_salary IS NOT NULL
GROUP BY ci.industry, jp.title
QUALIFY rang <= 10
ORDER BY ci.industry, salaire_moyen DESC;

-- Table 3 : Répartition des offres par taille d'entreprise
-- CASE WHEN transforme la valeur numérique (0-7) en label lisible
CREATE OR REPLACE TABLE linkedin.gold.repartition_taille_entreprise AS
SELECT
    CASE c.company_size
        WHEN 0 THEN '0 - Très petite'
        WHEN 1 THEN '1 - Petite'
        WHEN 2 THEN '2 - Petite-Moyenne'
        WHEN 3 THEN '3 - Moyenne'
        WHEN 4 THEN '4 - Moyenne-Grande'
        WHEN 5 THEN '5 - Grande'
        WHEN 6 THEN '6 - Très Grande'
        WHEN 7 THEN '7 - Géante'
        ELSE 'Inconnue'
    END AS taille_entreprise,
    COUNT(*) AS nb_offres
FROM linkedin.silver.job_postings jp
JOIN linkedin.silver.companies c
    ON jp.company_name = c.company_id
GROUP BY c.company_size
ORDER BY nb_offres DESC;

-- Table 4 : Répartition des offres par secteur d'activité
-- LIMIT 20 : garde uniquement les 20 secteurs les plus représentés
CREATE OR REPLACE TABLE linkedin.gold.repartition_secteur AS
SELECT
    ci.industry AS secteur,
    COUNT(*) AS nb_offres
FROM linkedin.silver.job_postings jp
JOIN linkedin.silver.company_industries ci
    ON jp.company_name = ci.company_id
GROUP BY ci.industry
ORDER BY nb_offres DESC
LIMIT 20;

-- Table 5 : Répartition des offres par type d'emploi
-- formatted_work_type : Full-time, Part-time, Contract, Internship, Temporary, Other
CREATE OR REPLACE TABLE linkedin.gold.repartition_type_emploi AS
SELECT
    formatted_work_type AS type_emploi,
    COUNT(*) AS nb_offres
FROM linkedin.silver.job_postings
WHERE formatted_work_type IS NOT NULL
GROUP BY formatted_work_type
ORDER BY nb_offres DESC;

-- Vérification : compter les lignes dans chaque table Gold
SELECT 'top_titres_par_industrie' AS table_name, COUNT(*) AS nb_lignes FROM linkedin.gold.top_titres_par_industrie
UNION ALL SELECT 'top_salaires_par_industrie', COUNT(*) FROM linkedin.gold.top_salaires_par_industrie
UNION ALL SELECT 'repartition_taille_entreprise', COUNT(*) FROM linkedin.gold.repartition_taille_entreprise
UNION ALL SELECT 'repartition_secteur', COUNT(*) FROM linkedin.gold.repartition_secteur
UNION ALL SELECT 'repartition_type_emploi', COUNT(*) FROM linkedin.gold.repartition_type_emploi;

---
## ETAPE 10 - Requêtes SQL des 5 analyses

In [ ]:
-- Analyse 1 - Top 10 des titres de postes les plus publiés par industrie
-- Les données sont déjà dans la table Gold, on lit directement
SELECT * FROM linkedin.gold.top_titres_par_industrie
ORDER BY nb_offres DESC
LIMIT 10;

In [ ]:
-- Analyse 2 - Top 10 des postes les mieux rémunérés par industrie
SELECT * FROM linkedin.gold.top_salaires_par_industrie
ORDER BY salaire_moyen DESC
LIMIT 10;

In [ ]:
-- Analyse 3 - Répartition des offres par taille d'entreprise
SELECT * FROM linkedin.gold.repartition_taille_entreprise
ORDER BY nb_offres DESC;

In [ ]:
-- Analyse 4 - Répartition des offres par secteur d'activité
SELECT * FROM linkedin.gold.repartition_secteur
ORDER BY nb_offres DESC
LIMIT 20;

In [ ]:
-- Analyse 5 - Répartition des offres par type d'emploi
SELECT * FROM linkedin.gold.repartition_type_emploi
ORDER BY nb_offres DESC;

---
## ETAPE 11 - Automatisation avec Snowpipe et tâches planifiées

Snowpipe surveille automatiquement le bucket S3.
Quand un nouveau fichier arrive, il est chargé automatiquement dans Bronze.
Les tâches planifiées propagent ensuite les données vers Silver et Gold.

Note : AUTO_INGEST = TRUE nécessite une configuration AWS (S3 → SQS)
non disponible sur un bucket public. Les pipes sont créés pour la démonstration.

In [ ]:
USE SCHEMA linkedin.bronze;

-- Creation des pipes Snowpipe pour chaque fichier
-- AUTO_INGEST = TRUE : chargement automatique dès qu'un fichier arrive dans S3

CREATE OR REPLACE PIPE linkedin.bronze.pipe_job_postings
    AUTO_INGEST = TRUE
AS
COPY INTO linkedin.bronze.job_postings
FROM @linkedin.bronze.s3_stage/job_postings.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.csv_format');

CREATE OR REPLACE PIPE linkedin.bronze.pipe_benefits
    AUTO_INGEST = TRUE
AS
COPY INTO linkedin.bronze.benefits
FROM @linkedin.bronze.s3_stage/benefits.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.csv_format');

CREATE OR REPLACE PIPE linkedin.bronze.pipe_employee_counts
    AUTO_INGEST = TRUE
AS
COPY INTO linkedin.bronze.employee_counts
FROM @linkedin.bronze.s3_stage/employee_counts.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.csv_format');

CREATE OR REPLACE PIPE linkedin.bronze.pipe_job_skills
    AUTO_INGEST = TRUE
AS
COPY INTO linkedin.bronze.job_skills
FROM @linkedin.bronze.s3_stage/job_skills.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.csv_format');

CREATE OR REPLACE PIPE linkedin.bronze.pipe_companies
    AUTO_INGEST = TRUE
AS
COPY INTO linkedin.bronze.companies
FROM @linkedin.bronze.s3_stage/companies.json
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.json_format');

CREATE OR REPLACE PIPE linkedin.bronze.pipe_company_industries
    AUTO_INGEST = TRUE
AS
COPY INTO linkedin.bronze.company_industries
FROM @linkedin.bronze.s3_stage/company_industries.json
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.json_format');

CREATE OR REPLACE PIPE linkedin.bronze.pipe_company_specialities
    AUTO_INGEST = TRUE
AS
COPY INTO linkedin.bronze.company_specialities
FROM @linkedin.bronze.s3_stage/company_specialities.json
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.json_format');

CREATE OR REPLACE PIPE linkedin.bronze.pipe_job_industries
    AUTO_INGEST = TRUE
AS
COPY INTO linkedin.bronze.job_industries
FROM @linkedin.bronze.s3_stage/job_industries.json
FILE_FORMAT = (FORMAT_NAME = 'linkedin.bronze.json_format');

-- Vérification : liste tous les pipes créés
SHOW PIPES IN SCHEMA linkedin.bronze;

-- Récupération du ARN SQS pour la configuration AWS
SELECT SYSTEM$PIPE_STATUS('linkedin.bronze.pipe_job_postings');

In [ ]:
USE SCHEMA linkedin.silver;

-- Tâche planifiée qui rafraichit Silver toutes les 60 minutes
-- Elle recharge job_postings depuis Bronze avec toutes les transformations
CREATE OR REPLACE TASK linkedin.silver.refresh_silver
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '60 MINUTE'
AS
BEGIN
    TRUNCATE TABLE linkedin.silver.job_postings;
    INSERT INTO linkedin.silver.job_postings
    SELECT
        job_id,
        SPLIT_PART(company_name, '.', 1) AS company_name,
        title, description,
        TRY_CAST(max_salary AS FLOAT),
        TRY_CAST(med_salary AS FLOAT),
        TRY_CAST(min_salary AS FLOAT),
        pay_period, formatted_work_type, location,
        TRY_CAST(applies AS INT),
        TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(original_listed_time) / 1000 AS BIGINT)),
        CASE WHEN remote_allowed = '1.0' THEN TRUE ELSE FALSE END,
        TRY_CAST(views AS INT),
        job_posting_url, application_url, application_type,
        TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(expiry) / 1000 AS BIGINT)),
        TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(closed_time) / 1000 AS BIGINT)),
        formatted_experience_level, skills_desc,
        TO_TIMESTAMP_NTZ(CAST(TRY_TO_DOUBLE(listed_time) / 1000 AS BIGINT)),
        posting_domain,
        CASE WHEN sponsored = '1' THEN TRUE ELSE FALSE END,
        work_type, currency, compensation_type
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY job_id) AS rn
        FROM linkedin.bronze.job_postings
    )
    WHERE rn = 1;
END;

USE SCHEMA linkedin.gold;

-- Tâche planifiée qui rafraichit Gold toutes les 70 minutes
-- Décalée de 10 minutes pour s'exécuter après Silver
CREATE OR REPLACE TASK linkedin.gold.refresh_gold
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '70 MINUTE'
AS
BEGIN
    CREATE OR REPLACE TABLE linkedin.gold.top_titres_par_industrie AS
    SELECT ci.industry, jp.title, COUNT(*) AS nb_offres,
        ROW_NUMBER() OVER (PARTITION BY ci.industry ORDER BY COUNT(*) DESC) AS rang
    FROM linkedin.silver.job_postings jp
    JOIN linkedin.silver.company_industries ci ON jp.company_name = ci.company_id
    GROUP BY ci.industry, jp.title
    QUALIFY rang <= 10
    ORDER BY ci.industry, nb_offres DESC;

    CREATE OR REPLACE TABLE linkedin.gold.top_salaires_par_industrie AS
    SELECT ci.industry, jp.title,
        ROUND(AVG(jp.max_salary), 2) AS salaire_moyen,
        ROW_NUMBER() OVER (PARTITION BY ci.industry ORDER BY AVG(jp.max_salary) DESC) AS rang
    FROM linkedin.silver.job_postings jp
    JOIN linkedin.silver.company_industries ci ON jp.company_name = ci.company_id
    WHERE jp.max_salary IS NOT NULL
    GROUP BY ci.industry, jp.title
    QUALIFY rang <= 10
    ORDER BY ci.industry, salaire_moyen DESC;

    CREATE OR REPLACE TABLE linkedin.gold.repartition_taille_entreprise AS
    SELECT
        CASE c.company_size
            WHEN 0 THEN '0 - Très petite'
            WHEN 1 THEN '1 - Petite'
            WHEN 2 THEN '2 - Petite-Moyenne'
            WHEN 3 THEN '3 - Moyenne'
            WHEN 4 THEN '4 - Moyenne-Grande'
            WHEN 5 THEN '5 - Grande'
            WHEN 6 THEN '6 - Très Grande'
            WHEN 7 THEN '7 - Géante'
            ELSE 'Inconnue'
        END AS taille_entreprise,
        COUNT(*) AS nb_offres
    FROM linkedin.silver.job_postings jp
    JOIN linkedin.silver.companies c ON jp.company_name = c.company_id
    GROUP BY c.company_size
    ORDER BY nb_offres DESC;

    CREATE OR REPLACE TABLE linkedin.gold.repartition_secteur AS
    SELECT ci.industry AS secteur, COUNT(*) AS nb_offres
    FROM linkedin.silver.job_postings jp
    JOIN linkedin.silver.company_industries ci ON jp.company_name = ci.company_id
    GROUP BY ci.industry
    ORDER BY nb_offres DESC
    LIMIT 20;

    CREATE OR REPLACE TABLE linkedin.gold.repartition_type_emploi AS
    SELECT formatted_work_type AS type_emploi, COUNT(*) AS nb_offres
    FROM linkedin.silver.job_postings
    WHERE formatted_work_type IS NOT NULL
    GROUP BY formatted_work_type
    ORDER BY nb_offres DESC;
END;

-- Activation des tâches planifiées
ALTER TASK linkedin.silver.refresh_silver RESUME;
ALTER TASK linkedin.gold.refresh_gold RESUME;

-- Vérification : liste toutes les tâches actives
SHOW TASKS IN DATABASE linkedin;

---
## ETAPE 12 - Code Streamlit

Ce code est à coller dans une Streamlit App Snowflake (Run on warehouse).
Il lit directement depuis les tables Gold pré-agrégées.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

# Connexion à la session Snowflake active
session = get_active_session()

# =============================================
# Analyse 1 - Top 10 des titres les plus publiés par industrie
# Lecture depuis la table Gold pré-agrégée
# Le filtre par industrie se fait côté Python avec selectbox
# =============================================
st.title('Analyse 1 - Top 10 des titres de postes les plus publiés par industrie')

df = session.sql('SELECT * FROM linkedin.gold.top_titres_par_industrie ORDER BY nb_offres DESC').to_pandas()

# Menu déroulant pour sélectionner une industrie
industries = sorted(df['INDUSTRY'].unique().tolist())
industrie_choisie = st.selectbox('Sélectionne une industrie :', industries, key='sel1')

# Filtrage et affichage du top 10 pour l'industrie choisie
df_filtre = df[df['INDUSTRY'] == industrie_choisie].sort_values('NB_OFFRES', ascending=False).head(10)
st.subheader(f'Top 10 des postes - {industrie_choisie}')
st.bar_chart(data=df_filtre.set_index('TITLE')['NB_OFFRES'])
st.subheader('Détail')
st.dataframe(df_filtre[['TITLE', 'NB_OFFRES']].reset_index(drop=True))


# =============================================
# Analyse 2 - Top 10 des postes les mieux rémunérés par industrie
# SALAIRE_MOYEN calculé avec AVG(max_salary) dans la table Gold
# =============================================
st.divider()
st.title('Analyse 2 - Top 10 des postes les mieux rémunérés par industrie')

df2 = session.sql('SELECT * FROM linkedin.gold.top_salaires_par_industrie ORDER BY salaire_moyen DESC').to_pandas()

industries2 = sorted(df2['INDUSTRY'].unique().tolist())
industrie_choisie2 = st.selectbox('Sélectionne une industrie :', industries2, key='sel2')

df_filtre2 = df2[df2['INDUSTRY'] == industrie_choisie2].sort_values('SALAIRE_MOYEN', ascending=False).head(10)
st.subheader(f'Top 10 salaires - {industrie_choisie2}')
st.bar_chart(data=df_filtre2.set_index('TITLE')['SALAIRE_MOYEN'])
st.subheader('Détail')
st.dataframe(df_filtre2[['TITLE', 'SALAIRE_MOYEN']].reset_index(drop=True))


# =============================================
# Analyse 3 - Répartition des offres par taille d'entreprise
# Pas de filtre nécessaire, on affiche toutes les tailles
# =============================================
st.divider()
st.title('Analyse 3 - Répartition des offres par taille d\'entreprise')

df3 = session.sql('SELECT * FROM linkedin.gold.repartition_taille_entreprise ORDER BY nb_offres DESC').to_pandas()

st.subheader('Répartition des offres par taille d\'entreprise')
st.bar_chart(data=df3.set_index('TAILLE_ENTREPRISE')['NB_OFFRES'])
st.subheader('Détail')
st.dataframe(df3)


# =============================================
# Analyse 4 - Répartition des offres par secteur d'activité
# Top 20 des secteurs les plus représentés
# =============================================
st.divider()
st.title('Analyse 4 - Répartition des offres par secteur d\'activité')

df4 = session.sql('SELECT * FROM linkedin.gold.repartition_secteur ORDER BY nb_offres DESC').to_pandas()

st.subheader('Top 20 des secteurs par nombre d\'offres')
st.bar_chart(data=df4.set_index('SECTEUR')['NB_OFFRES'])
st.subheader('Détail')
st.dataframe(df4)


# =============================================
# Analyse 5 - Répartition des offres par type d'emploi
# Types : Full-time, Part-time, Contract, Internship, Temporary, Other
# =============================================
st.divider()
st.title('Analyse 5 - Répartition des offres par type d\'emploi')

df5 = session.sql('SELECT * FROM linkedin.gold.repartition_type_emploi ORDER BY nb_offres DESC').to_pandas()

st.subheader('Répartition par type d\'emploi')
st.bar_chart(data=df5.set_index('TYPE_EMPLOI')['NB_OFFRES'])
st.subheader('Détail')
st.dataframe(df5[['TYPE_EMPLOI', 'NB_OFFRES']].reset_index(drop=True))